# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) dataset using the `mlcroissant` library and the Croissant schema.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

This notebook guides you through:
- Loading the dataset via its Croissant schema
- Reviewing available record sets and fields (by their `@id`s)
- Extracting data into DataFrames
- Performing basic exploratory data analysis (EDA)
- Visualizing relationships between fields

In [ ]:
# Install and upgrade mlcroissant if needed
!pip install --quiet --upgrade mlcroissant

## 1. Data Loading
Load metadata and browse the available record sets using the Croissant schema. The `mlcroissant` library provides a simple API for this.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset's Croissant metadata
dataset = mlc.Dataset(croissant_url)

# Access key metadata fields
meta = dataset.metadata
print(f"Dataset: {meta.name}\n")
print(f"Description: {meta.description}\n")
print(f"Version: {meta.version}\nPublished: {meta.datePublished}\n")
print(f"License: {meta.license}")

## 2. Data Overview
Review the dataset's record sets, their fields, and IDs. 
You should reference all record sets, fields, and columns by their Croissant `@id`.

Let's enumerate the record sets:

In [ ]:
# Each RecordSet object in dataset.record_sets has attributes: id, name, description, fields, columns, etc.

print("Available record sets:")
for rs in dataset.record_sets:
    print(f"  - Name: {getattr(rs, 'name', 'N/A')}")
    print(f"    @id: {rs.id}")
    if hasattr(rs, 'description'):
        print(f"    Description: {rs.description}")

    # List fields and their ids
    fields = getattr(rs, 'fields', [])
    if fields:
        print("    Fields:")
        for f in fields:
            print(f"      - {getattr(f, 'name', 'N/A')} (@id: {f.id})")
    # List columns and their ids if any
    columns = getattr(rs, 'columns', [])
    if columns:
        print("    Columns:")
        for c in columns:
            print(f"      - {getattr(c, 'name', 'N/A')} (@id: {c.id})")
    print()
if not dataset.record_sets:
    print("No record sets defined directly in schema. Attempting to load from data distributions...")

### Attempt to automatically detect and list record sets
If no `RecordSet` objects are present as top-level entities, try to view available resource IDs from the distribution list for further exploration.

You can optionally try manually specifying a known `record_set` `@id` from data files.

In [ ]:
meta_json = dataset.metadata.to_json()
if 'distribution' in meta_json:
    print("Distributions (potential resources/data files):")
    for dist in meta_json['distribution']:
        print(f"  @id: {dist['@id']}")
else:
    print("No distribution entries found.")

## 3. Data Extraction
Extract records from a given record set by its ID. In this dataset, record sets must be referenced via their `@id`s. If no record sets are present, 
try available distribution IDs as record set IDs (Croissant-compatible datasets may use distribution `@id`s as record set IDs if not otherwise defined).

In [ ]:
# Example distribution/record set @id(s) from the schema:
record_set_ids = [
    "http://nexus-delta.data-vitae-prd.svc.cluster.local/v1/resources/frontiers/7853015/_/8336ac61-9308-403f-8df3-28e120cc98f3",
    "http://nexus-delta.data-vitae-prd.svc.cluster.local/v1/resources/frontiers/7853015/_/8e507442-660d-4cfe-b2d9-f805d7abe725"
]

dataframes = {}
for rid in record_set_ids:
    try:
        records = list(dataset.records(record_set=rid))
        df = pd.DataFrame(records)
        dataframes[rid] = df
        print(f"Loaded: {rid} | Rows: {len(df)} | Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Failed to load records for record set @id {rid}: {e}")

# Pick the first non-empty DataFrame for demonstration:
main_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rid
        break

if main_record_set_id:
    print(f"\nMain record set (@id): {main_record_set_id}")
    print(f"Columns: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No non-empty record sets found.")

## 4. Exploratory Data Analysis (EDA)
Let's perform basic EDA using field `@id`s. We'll: 
- Filter by a numeric column if available (by @id)
- Normalize that column
- Group by a categorical column (by @id)

> **Always reference columns/fields using their Croissant `@id` (not name), if available.**

In [ ]:
# Check if main data frame exists and is non-empty
if main_record_set_id:
    df = dataframes[main_record_set_id]
    
    # List of all columns (by name)
    print("Columns present (column names):\n", df.columns.tolist())
    
    # Guess likely numeric field @id (try for 'log_likelihood', 'p_value' etc.)
    # For illustration, pick the first column with a numeric dtype
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if not numeric_field_id:
        print("No numeric columns found in this record set.")
    else:
        print(f"Using `{numeric_field_id}` for numeric EDA.")

        # Filtering
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        col_norm = numeric_field_id + '_normalized'
        filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, col_norm]].head())

        # Find a categorical field for grouping (string/object, with low cardinality)
        group_field = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and df[col].nunique() < max(10, int(0.1*len(df))):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"\nGrouped mean {numeric_field_id} by {group_field} (by @id):")
            display(grouped_df)
        else:
            print("No suitable categorical field found for grouping.")
else:
    print("No main data frame available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and, if grouped, its mean by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id} (by @id)")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If grouping was performed, make a barplot
    if 'grouped_df' in locals() and group_field and not grouped_df.empty:
        grouped_df.plot(kind='bar', figsize=(8,4))
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field)
        plt.tight_layout()
        plt.show()
else:
    print("Visualization skipped: numeric field or data frame not found.")

## 6. Conclusion
In this notebook, you have seen how to:
- Load a Croissant-format dataset with `mlcroissant` using its schema URL
- Inspect metadata, record sets, fields, and their `@id`s
- Extract structured records from a record set referenced by its `@id`
- Explore, filter, normalize, and group data in a DataFrame
- Visualize numeric variable distributions and grouped summaries

For further exploration, reference all dataset entities using their `@id`s for rigorous reproducibility.

<sub>Notebook generated for dataset: [10.71728/senscience.y7m0-f273](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)</sub>